In [1]:
# import os
# import wandb
# from kaggle_secrets import UserSecretsClient

# # fetch secret
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# # set environment variable
# os.environ["WANDB_API_KEY"] = wandb_key

# # login
# wandb.login(relogin=True)

In [2]:
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 73.9 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:
      Successfully uninstalled wandb-0.22.2


In [3]:
import os
import wandb

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
    wandb.login(key=wandb_key)
    print("✓ W&B authenticated via Kaggle secrets")
    
except Exception as e:
    print(f"⚠ W&B login failed: {e}")
    print("Proceeding without W&B logging...")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


✓ W&B authenticated via Kaggle secrets


In [4]:
from pathlib import Path

MESSY_DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
ESC_50_MASTER = MESSY_DATA_DIR / "ESC-50-master"
GENRES_STEMS = MESSY_DATA_DIR / "genres_stems"
MASHUPS = MESSY_DATA_DIR / "mashups"

In [5]:
import os
os.environ["TORCHAUDIO_USE_BACKEND_DISPATCHER"] = "0"

import random
from pathlib import Path
from torch.utils.data import Dataset
import torchaudio
from transformers import Wav2Vec2Processor
import torch
import numpy as np
from scipy.signal import butter, filtfilt


def highpass_filter(audio: torch.Tensor, cutoff: int = 40, sample_rate: int = 16000):
    b, a = butter(5, cutoff / (sample_rate / 2), btype='high')

    audio_np = audio.squeeze(0).cpu().numpy()
    filtered = filtfilt(b, a, audio_np)

    filtered = np.ascontiguousarray(filtered)

    filtered = torch.from_numpy(filtered).unsqueeze(0)
    return filtered.type_as(audio)


class Audio_DataLoader(Dataset):

    def __init__(self, genre_dir: Path, esc50_dir: Path):

        self.genre_dir = genre_dir
        self.esc50_paths = list(esc50_dir.rglob("*.wav"))

        self.SAMPLE_RATE = 16000
        self.CHUNK_DURATION = 8
        self.CHUNK_SAMPLES = self.SAMPLE_RATE * self.CHUNK_DURATION

        self.samples = []
        self.labels = []
        self.class_map = {}

        self._load_files()
        self._build_index_map()

    def _load_files(self):

        for label_id, genre_dir in enumerate(sorted(self.genre_dir.iterdir())):

            if not genre_dir.is_dir():
                continue

            genre_name = genre_dir.name
            self.class_map[genre_name] = label_id

            for song_dir in genre_dir.iterdir():

                if not song_dir.is_dir():
                    continue

                wav_parts = sorted(list(song_dir.glob("*.wav")))
                if len(wav_parts) == 0:
                    continue

                self.samples.append(wav_parts)
                self.labels.append(label_id)

    def _build_index_map(self):

        self.index_map = []

        for song_idx in range(len(self.samples)):

            # load only first stem to know duration
            waveform, sr = torchaudio.load(self.samples[song_idx][0])
            waveform = waveform.mean(dim=0, keepdim=True)

            if sr != self.SAMPLE_RATE:
                resampler = torchaudio.transforms.Resample(sr, self.SAMPLE_RATE)
                waveform = resampler(waveform)

            total_len = waveform.shape[1]

            n_chunks = (total_len + self.CHUNK_SAMPLES - 1) // self.CHUNK_SAMPLES

            for c in range(n_chunks):
                start = c * self.CHUNK_SAMPLES
                self.index_map.append((song_idx, start))

        print("Total training segments:", len(self.index_map))

    def __len__(self):
        return len(self.index_map)

    def merge_song(self, wave_list):

        waveforms = []

        for wav_path in wave_list:
            waveform, sr = torchaudio.load(wav_path)

            waveform = waveform.mean(dim=0, keepdim=True)

            if sr != self.SAMPLE_RATE:
                resampler = torchaudio.transforms.Resample(sr, self.SAMPLE_RATE)
                waveform = resampler(waveform)

            waveform = highpass_filter(waveform, sample_rate=self.SAMPLE_RATE)

            waveforms.append(waveform)

        max_len = max(w.shape[1] for w in waveforms)

        padded = []
        for w in waveforms:
            if w.shape[1] < max_len:
                w = torch.nn.functional.pad(w, (0, max_len - w.shape[1]))
            padded.append(w)

        full_waveform = torch.stack(padded, dim=0).mean(dim=0)

        return full_waveform

    def get_chunk(self, waveform: torch.Tensor, start: int):

        end = start + self.CHUNK_SAMPLES

        if waveform.shape[1] < end:
            padding = end - waveform.shape[1]
            chunk = torch.nn.functional.pad(waveform[:, start:], (0, padding))
        else:
            chunk = waveform[:, start:end]

        return chunk

    def inject_esc50_noise(self, music_waveform):

        if random.random() > 0.8:
            return music_waveform

        samples = music_waveform.shape[1]
        n_events = random.randint(1, 2)

        for _ in range(n_events):

            noise_path = random.choice(self.esc50_paths)
            noise_wave, sr = torchaudio.load(noise_path)

            noise_wave = noise_wave.mean(dim=0, keepdim=True)

            if sr != self.SAMPLE_RATE:
                resampler = torchaudio.transforms.Resample(sr, self.SAMPLE_RATE)
                noise_wave = resampler(noise_wave)

            noise_len = noise_wave.shape[1]

            if noise_len >= samples:
                noise_wave = noise_wave[:, :samples]
                start = 0
            else:
                start = random.randint(0, samples - noise_len)

            snr_db = random.uniform(8, 18)

            music_power = music_waveform.pow(2).mean()
            noise_power = noise_wave.pow(2).mean() + 1e-9

            scale = torch.sqrt(music_power / (noise_power * (10 ** (snr_db / 10))))
            music_waveform[:, start:start+noise_len] += scale * noise_wave

        return music_waveform

    def __getitem__(self, index):
        song_idx, start = self.index_map[index]
        
        wave_list = self.samples[song_idx]
        label = self.labels[song_idx]

        full_waveform = self.merge_song(wave_list)
        full_waveform = self.get_chunk(full_waveform, start)
        full_waveform = self.inject_esc50_noise(full_waveform)

        rms = torch.sqrt(torch.mean(full_waveform**2) + 1e-9)
        full_waveform = full_waveform / (rms * 3.0)
        full_waveform = full_waveform.clamp(-1.0, 1.0)
        full_waveform = full_waveform.squeeze(0)

        return full_waveform, torch.tensor(label, dtype=torch.long)


processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

def collate_fn(batch):
    """
    Collate function for Wav2Vec2
    Converts list of (waveform, label) → batched tensors
    """
    waveforms = []
    labels = []

    for waveform, label in batch:
        waveform = waveform.cpu().numpy()
        waveforms.append(waveform)
        labels.append(label.item() if isinstance(label, torch.Tensor) else label)

    # Wav2Vec2Processor handles padding and attention mask
    inputs = processor(
        waveforms,
        sampling_rate=16000,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.long)

    return inputs.input_values, inputs.attention_mask, labels

2026-02-21 04:49:55.519436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771649395.701206      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771649395.753567      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771649396.209197      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771649396.209244      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771649396.209247      24 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [6]:
import torch 
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time
import wandb


def train_wav2vec2(model, dataloader, loss_function, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for input_values, attention_mask, labels in dataloader:
        input_values = input_values.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        logits = model(input_values, attention_mask)
        
        loss = loss_function(logits, labels)
        
        if torch.isnan(loss):
            print(f"⚠ NaN loss detected, skipping batch")
            optimizer.zero_grad()
            continue

        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


def test_wav2vec2(model, dataloader, loss_function, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_values, attention_mask, labels in dataloader:
            input_values = input_values.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            logits = model(input_values, attention_mask)
            
            loss = loss_function(logits, labels)

            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0
    accuracy = 100 * correct / total if total > 0 else 0

    return loss, accuracy


def train_wav2vec2_model(model, device, train_loader, val_loader, epochs=30):
    
    print("\n" + "="*70)
    print("WAV2VEC2 GENRE CLASSIFICATION TRAINING")
    print("="*70 + "\n")
    
    if device == "cuda":
        torch.cuda.empty_cache()

    wandb.init(
        project="wav2vec2-genre-classification",
        name="3-stage-transfer-learning",
        config={
            "epochs": epochs,
            "optimizer": "Adam",
            "dataset": "Genre",
            "architecture": model.__class__.__name__,
            "base_model": "wav2vec2-base",
        }
    )
    wandb.watch(model, log="all", log_freq=50)

    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    
    start_time = time.time()
    best_val_acc = 0
    best_val_loss = float('inf')
    
    change_stage = epochs // 3  

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
        weight_decay=1e-4
    )
    
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )

    for epoch in range(epochs):
        epoch_start = time.time()
        
        if epoch == 0:
            print(f"STAGE 1: Training only classifier head (lr=1e-3)")
            
            # Freeze entire wav2vec encoder
            for param in model.wav2vec.parameters():
                param.requires_grad = False
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=1e-3,
                weight_decay=1e-4
            )
            scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
            
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in model.parameters())
            print(f"         Trainable: {trainable:,} / {total:,}")
            print(f"{'='*70}\n")
            
            wandb.log({"stage": "classifier_only"}, step=epoch)

        elif epoch == change_stage:
            print(f"\n{'='*70}")
            print(f"STAGE 2: Unfroze top 2 encoder layers (lr=5e-5)")
            
            for name, param in model.wav2vec.named_parameters():
                if "encoder.layers.10" in name or "encoder.layers.11" in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=5e-5,
                weight_decay=1e-4
            )
            scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
            
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in model.parameters())
            print(f"         Trainable: {trainable:,} / {total:,}")
            print(f"{'='*70}\n")
            
            wandb.log({"stage": "partial_unfreeze"}, step=epoch)

        elif epoch == change_stage * 2:
            print(f"\n{'='*70}")
            print(f"STAGE 3: Full model fine-tuning (lr=1e-5)")
            for name, param in model.wav2vec.named_parameters():
                if "feature_extractor" not in name:
                    param.requires_grad = True
            
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=1e-5,
                weight_decay=1e-4
            )
            scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
            
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in model.parameters())
            print(f"         Trainable: {trainable:,} / {total:,}")
            print(f"{'='*70}\n")
            
            wandb.log({"stage": "full_unfreeze"}, step=epoch)

        train_loss, train_acc = train_wav2vec2(
            model, train_loader, loss_function, optimizer, device
        )
        
        val_loss, val_acc = test_wav2vec2(
            model, val_loader, loss_function, device
        )
        
        scheduler.step(val_loss)
        
        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": current_lr,
            "epoch_time": epoch_time,
        })

        print(
            f"Epoch [{epoch+1:3d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:6.2f}% | "
            f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}s"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_wav2vec2_genre.pth")
            wandb.save("best_wav2vec2_genre.pth")
            print(f"         ✓ Saved best model (val_acc: {val_acc:.2f}%, val_loss: {val_loss:.4f})")

    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"WAV2VEC2 TRAINING COMPLETE")
    print(f"{'='*70}")
    print(f"Total Training Time    : {total_time/60:.1f} minutes")
    print(f"Best Validation Acc    : {best_val_acc:.2f}%")
    print(f"Best Validation Loss   : {best_val_loss:.4f}")
    print(f"Model saved as         : best_wav2vec2_genre.pth")
    print(f"{'='*70}\n")

    wandb.finish()
    
    return best_val_acc


In [7]:
! pip install torchlibrosa

In [8]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class Wav2Vec2ForGenre(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # load pretrained encoder
        self.wav2vec = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base"
        )

        self.wav2vec.feature_extractor._freeze_parameters()

        hidden_size = self.wav2vec.config.hidden_size  # 768

        # classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_values, attention_mask):

        # encoder output
        outputs = self.wav2vec(
            input_values=input_values,
            attention_mask=attention_mask
        )

        # shape: (B, T, 768)
        hidden_states = outputs.last_hidden_state

        # wav2vec outputs time sequence features
        # we must convert sequence → single vector
        pooled = hidden_states.mean(dim=1)

        logits = self.classifier(pooled)

        return logits


In [9]:
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = 10   # genres
model = Wav2Vec2ForGenre(num_classes).to(device)

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

In [10]:
from torch.utils.data import WeightedRandomSampler, DataLoader, Subset
from collections import Counter
import numpy as np

dataset = Audio_DataLoader(
    genre_dir=GENRES_STEMS,
    esc50_dir=ESC_50_MASTER
)
num_songs = len(dataset.samples)
song_indices = np.arange(num_songs)
np.random.seed(42)
np.random.shuffle(song_indices)

train_song_count = int(0.8 * num_songs)
train_songs = set(song_indices[:train_song_count])
val_songs = set(song_indices[train_song_count:])

print(f"Songs split: {len(train_songs)} train, {len(val_songs)} val")

train_indices = [i for i, (song_idx, start) in enumerate(dataset.index_map) 
                 if song_idx in train_songs]
val_indices = [i for i, (song_idx, start) in enumerate(dataset.index_map) 
               if song_idx in val_songs]

# Create subsets
train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

print(f"Train chunks: {len(train_dataset)}")
print(f"Val chunks: {len(val_dataset)}")


# Get labels for training chunks
train_labels = [dataset.labels[dataset.index_map[i][0]] for i in train_indices]

class_counts = Counter(train_labels)

class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nClass distribution in training chunks:")
genre_names = {v: k for k, v in dataset.class_map.items()}
for cls, count in sorted(class_counts.items()):
    print(f"  {genre_names[cls]}: {count} chunks (weight: {class_weights[cls]:.3f})")

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True if device == "cuda" else False
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True if device == "cuda" else False
)

print(f"\n{'='*70}")
print(f"DATALOADERS READY")
print(f"{'='*70}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")
print(f"{'='*70}\n")

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Total training segments: 4000
Songs split: 800 train, 200 val
Train chunks: 3200
Val chunks: 800

Class distribution in training chunks:
  blues: 304 chunks (weight: 0.003)
  classical: 332 chunks (weight: 0.003)
  country: 320 chunks (weight: 0.003)
  disco: 328 chunks (weight: 0.003)
  hiphop: 312 chunks (weight: 0.003)
  jazz: 312 chunks (weight: 0.003)
  metal: 336 chunks (weight: 0.003)
  pop: 292 chunks (weight: 0.003)
  reggae: 308 chunks (weight: 0.003)
  rock: 356 chunks (weight: 0.003)

DATALOADERS READY
Train batches: 200
Val batches: 50



In [11]:
 train_wav2vec2_model(
    model=model,
    device=device,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    epochs=30
)


WAV2VEC2 GENRE CLASSIFICATION TRAINING



wandb: setting up run 0lfdk8b6
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260221_045156-0lfdk8b6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 3-stage-transfer-learning
wandb: ⭐️ View project at https://wandb.ai/23f2003086-t12026-project/wav2vec2-genre-classification
wandb: 🚀 View run at https://wandb.ai/23f2003086-t12026-project/wav2vec2-genre-classification/runs/0lfdk8b6


STAGE 1: Training only classifier head (lr=1e-3)
         Trainable: 398,858 / 94,770,570



/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Epoch [  1/30] | Train Loss: 1.7094 | Train Acc:  41.31% | Val Loss: 1.5723 | Val Acc:  49.12% | LR: 1.00e-03 | Time: 1176.5s


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


         ✓ Saved best model (val_acc: 49.12%, val_loss: 1.5723)
Epoch [  2/30] | Train Loss: 1.2553 | Train Acc:  56.62% | Val Loss: 1.4300 | Val Acc:  52.88% | LR: 1.00e-03 | Time: 1030.1s
         ✓ Saved best model (val_acc: 52.88%, val_loss: 1.4300)
Epoch [  3/30] | Train Loss: 1.1366 | Train Acc:  60.50% | Val Loss: 1.4737 | Val Acc:  53.38% | LR: 1.00e-03 | Time: 1017.1s
         ✓ Saved best model (val_acc: 53.38%, val_loss: 1.4737)
Epoch [  4/30] | Train Loss: 1.0626 | Train Acc:  63.34% | Val Loss: 1.3151 | Val Acc:  57.12% | LR: 1.00e-03 | Time: 1035.2s
         ✓ Saved best model (val_acc: 57.12%, val_loss: 1.3151)
Epoch [  5/30] | Train Loss: 1.0092 | Train Acc:  65.00% | Val Loss: 1.3213 | Val Acc:  57.50% | LR: 1.00e-03 | Time: 1002.7s
         ✓ Saved best model (val_acc: 57.50%, val_loss: 1.3213)
Epoch [  6/30] | Train Loss: 0.9571 | Train Acc:  67.06% | Val Loss: 1.3675 | Val Acc:  56.88% | LR: 1.00e-03 | Time: 990.9s
Epoch [  7/30] | Train Loss: 0.9216 | Train Acc:  6

wandb: updating run metadata


         ✓ Saved best model (val_acc: 76.75%, val_loss: 1.1173)

WAV2VEC2 TRAINING COMPLETE
Total Training Time    : 534.1 minutes
Best Validation Acc    : 76.75%
Best Validation Loss   : 1.1173
Model saved as         : best_wav2vec2_genre.pth



wandb: uploading best_wav2vec2_genre.pth; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading best_wav2vec2_genre.pth; uploading wandb-summary.json
wandb: uploading best_wav2vec2_genre.pth
wandb: uploading best_wav2vec2_genre.pth; uploading history steps 29-29, summary, console lines 67-77
wandb: uploading best_wav2vec2_genre.pth
wandb: 
wandb: Run history:
wandb:          epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:     epoch_time █▂▂▃▁▁▂▂▂▂▂▂▂▂▂▂▂▁▁▂█▇██▇▇▇█▇▇
wandb:  learning_rate ███████▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_accuracy ▁▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▅▅▆▆▇▇▇████
wandb:     train_loss █▆▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▂▂▂▁▁▁▁
wandb:   val_accuracy ▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▄▆▇▆▆▆▇▇▇██
wandb:       val_loss █▆▇▅▅▆▆▆▆▅▄▅▄▅▅▄▅▄▄▅▂▁▃▄▄▄▃▄▃▃
wandb: 
wandb: Run summary:
wandb:          epoch 30
wandb:     epoch_time 1136.96463
wandb:  learning_rate 0.0
wandb:          stage full_unfreeze
wandb: train_accuracy 95.0625
wandb:     train_loss 0.14755
wandb:   val_accuracy 

76.75